# Optical / multispectral — Sentinel-2 SR Harmonized

Sentinel-2 Surface Reflectance Harmonized — June 2024 median of the red band (`B4`) over Giza at 30 m. Median reduction is the canonical cloud-screened composite.

## Setup

Imports and a per-notebook output directory. `pyramids` provides `Dataset` (GeoTIFF reading + plotting); `earthlens` provides the unified `EarthLens` entry point and the GEE `Catalog`.

In [ ]:
import os
from pathlib import Path

from pyramids.dataset import Dataset
from pyramids.plot import ColorBar

from earthlens.core import EarthLens
from earthlens.gee import Catalog, cancel_task

OUT_DIR = Path('out') / 'optical-multispectral'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'output directory: {OUT_DIR.resolve()}')

### Credentials

The notebook reads the GEE service-account credentials from the `GEE_SERVICE_ACCOUNT` / `GEE_SERVICE_KEY` environment variables. Both must be set before running the download cells below.

In [ ]:
SERVICE_ACCOUNT = os.environ['GEE_SERVICE_ACCOUNT']
SERVICE_KEY = os.environ['GEE_SERVICE_KEY']

## Inspect the catalog entry

Before downloading anything, look at what the bundled catalog knows about the asset — bands, cadence, license, provider.

In [ ]:
cat = Catalog()
ds = cat.get_dataset('COPERNICUS/S2_SR_HARMONIZED')
print(ds)

# The summary clips long text and shows only a band count, so an explorer
# notebook still wants the untruncated title and the fields it omits:
print(f'title (full):        {ds.title}')
print(f'ee_type:             {ds.ee_type}')
print(f'default_reducer:     {ds.default_reducer}')
print(f'license:             {ds.license}')
print(f'band ids (first 5):  {list(ds.bands)[:5]}')

## Download

Tiny AOI ([29.95, 30.05] lat, [31.15, 31.25] lon) at 30.0 m, `monthly` cadence — keeps the synchronous download under EE's 32768-px per-axis cap. The request and authentication are kept on separate lines so each step is easy to read and re-run.

In [ ]:
gee = EarthLens(
    data_source="gee",
    start='2024-06-01',
    end='2024-06-30',
    dataset='COPERNICUS/S2_SR_HARMONIZED',
    variables=['B4'],
    aoi=[31.15, 29.95, 31.25, 30.05],
    cadence='monthly',
    path=OUT_DIR,
    scale=30.0,
    reducer='median',
)
gee.authenticate(service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY)

Run the synchronous download. A live Earth Engine failure raises here rather than being recorded and passed
over, so a broken request stops the notebook instead of producing an empty one.

In [ ]:
paths = gee.download(progress_bar=False)
print(f'wrote {len(paths)} GeoTIFF(s):')
for p in paths:
    print(f'  {p}  ({p.stat().st_size / 1024:.1f} KB)')

## Quick preview

Load the first written GeoTIFF through pyramids and render the single band. (`pyramids.dataset.Dataset` is the project's GeoTIFF/NetCDF wrapper.) The dataset's nodata is masked so the colormap isn't pinned to it.

In [ ]:
preview = Dataset.read_file(paths[0])

# A few bright roofs stretch B4 to 6056 DN while the bulk of the scene sits near
# 2000, so the full range washes the image out. Two standard deviations either
# side of the mean is the usual single-band stretch, and stats() already carries
# both numbers.
stats = preview.stats(approx_ok=False)
mean, std = stats['mean'].iloc[0], stats['std'].iloc[0]
vmin, vmax = mean - 2 * std, mean + 2 * std

glyph = preview.plot(
    cmap='Greys_r',
    colorbar=ColorBar(label='B4 reflectance (DN)'),
    title='Sentinel-2 red band — Cairo and the Nile',
)
glyph.ax.title.set_fontsize(11)
glyph.im.set_clim(vmin, vmax)

low, high = stats['min'].iloc[0], stats['max'].iloc[0]
print(f'value range:     [{low:.4g}, {high:.4g}] DN')
print(f'display stretch: [{vmin:.0f}, {vmax:.0f}] DN (mean ± 2σ)')

## Tracking submitted jobs (asynchronous export)

The download above uses `export_via="url"` — a synchronous `getDownloadURL` round-trip. Nothing was queued, so there's no Earth Engine job to track.

To track an export instead, switch to an asynchronous sink (`drive` / `gcs` / `asset`) and pass `wait_for_export=False` so `.download()` returns a `TaskInfo` at submission time rather than blocking until completion. The cells below submit the same `(asset_id, band, AOI, scale)` request as an `export_via="asset"` task into the service account's own asset folder, then walk the four jobs-API calls (`list_recent_tasks` → `wait_for_task_id` → `ee.data.getAsset` → `ee.data.deleteAsset`) to make the job finish *and* tidy up. See `track-batch-exports.ipynb` for a deeper worked example.

### Prepare the demo asset folder

`GEE._export_via_batch` writes the image at `<asset_id>/<prefix>`, so `asset_id` here is the parent *folder*, not the final image path. Clear any leftover children from a previous run, then create the empty folder EE requires before a child write.

In [ ]:
import ee

from earthlens.gee import list_recent_tasks, wait_for_task_id

# The asset goes into a `Folder` asset that we own. `GEE._export_via_batch`
# writes the actual image at `<asset_id>/<prefix>`, so `asset_id` here is
# the parent FOLDER (not the final image path). Both must be cleaned up.
_proj = ee.data._get_projects_path().removeprefix('projects/')
PARENT = f'projects/{_proj}/assets'
DEMO_FOLDER = f'{PARENT}/earthlens-demo-optical-multispectral'
print(f'demo folder: {DEMO_FOLDER}')

# Listing the parent says whether a previous run left the folder behind, so the
# cleanup below never has to swallow a "not found" from Earth Engine.
listed = ee.data.listAssets({'parent': PARENT})
siblings = [asset['name'] for asset in listed.get('assets', [])]
if DEMO_FOLDER in siblings:
    children = ee.data.listAssets({'parent': DEMO_FOLDER})
    for child in children.get('assets', []):
        ee.data.deleteAsset(child['name'])
        print(f'cleared leftover child: {child["name"]}')
    ee.data.deleteAsset(DEMO_FOLDER)
    print(f'cleared leftover folder: {DEMO_FOLDER}')
# Create the parent folder — EE requires it to exist before a child write.
ee.data.createAsset({'type': 'Folder'}, DEMO_FOLDER)
print(f'created folder: {DEMO_FOLDER}')

### Submit

Same `(asset_id, band, AOI, scale)` request as the sync download above, just routed through `export_via="asset"` + `wait_for_export=False`. The construct → `authenticate` → `download` steps are kept on separate lines; `download()` returns a `TaskInfo` per submitted bucket at the moment the task is queued — no blocking.

In [ ]:
async_gee = EarthLens(
    data_source="gee",
    start='2024-06-01',
    end='2024-06-30',
    dataset='COPERNICUS/S2_SR_HARMONIZED',
    variables=['B4'],
    aoi=[31.15, 29.95, 31.25, 30.05],
    cadence='monthly',
    path=OUT_DIR,
    scale=30.0,
    reducer='median',
    export_via='asset',
    asset_id=DEMO_FOLDER,
    wait_for_export=False,
)
async_gee.authenticate(service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY)
submitted = async_gee.download(progress_bar=False)
task_info = submitted[0]
print(f'submitted: id={task_info.id} state={task_info.state}')
print(f'           description={task_info.description}')

### List + wait

`list_recent_tasks(description_prefix=...)` returns every matching task across the current project; `wait_for_task_id` blocks until the one we care about reaches a terminal state. A real workflow would just poll later from a separate process — the wait here exists so the notebook shows the full success path end-to-end.

In [ ]:
recent = list_recent_tasks(
    description_prefix=task_info.description,
    max_age_min=10,
)
print(f'list_recent_tasks matched {len(recent)} task(s):')
for t in recent:
    print(f'  {t.id}  {t.state:<12} {t.description}')
final = None
try:
    final = wait_for_task_id(
        task_info.id,
        poll_seconds=10,
        progress_bar=False,
    )
    print(f'\nfinal state: {final.state}')
finally:
    if final is None:
        # The wait raises on FAILED / CANCELLED *and on timeout* — and a
        # timeout leaves the export still running. Cancel it so an aborted
        # notebook does not leave a live task behind; cancel_task is a no-op
        # on an already-terminal task, and the original error still
        # propagates out of this finally.
        cancel_task(task_info.id)
        print(f'cancelled {task_info.id} after the wait failed')

### Verify + clean up

Confirm the produced asset exists on Earth Engine, then delete it (and the surrounding demo folder) so we don't leak storage between notebook runs. The backend wrote the image at `<DEMO_FOLDER>/<task description>`.

In [ ]:
produced = f'{DEMO_FOLDER}/{task_info.description}'
meta = ee.data.getAsset(produced)
print(f'asset exists: type={meta.get("type")} name={meta.get("name")}')
ee.data.deleteAsset(produced)
print('asset deleted')
# Tear down the parent folder.
ee.data.deleteAsset(DEMO_FOLDER)
print(f'folder deleted: {DEMO_FOLDER}')

## What's on disk

The written GeoTIFF is left under the per-notebook `out/` directory for you to inspect. That directory is
`.gitignore`d — re-running the notebook overwrites it.

In [ ]:
for p in sorted(OUT_DIR.iterdir()) if OUT_DIR.exists() else []:
    print(f'{p}  ({p.stat().st_size / 1024:.1f} KB)')